<a href="https://colab.research.google.com/github/AlexisCuevasUriostique/Minerva/blob/3-Fields-Architecture/Shai_Hulud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Tokenizer
import numpy as np
import os

# --- 1. ARCHITECTURE: PARALLAX & 3-FIELDS ---
class ParallaxAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

        # [cite_start]THE LENS (Subject Position / Anchor) [cite: 36, 56, 77]
        self.lens = nn.Parameter(torch.randn(1, 1, embed_dim))

    def forward(self, x, mask=None):
        B, T, C = x.shape
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # [cite_start]Apply Parallax: Q + Lens [cite: 37, 58, 79]
        q_parallax = q + self.lens

        q_parallax = q_parallax.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        scores = (q_parallax @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        return self.out_proj((attn_weights @ v).transpose(1, 2).contiguous().view(B, T, C))

class ParallaxGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, context_len=32):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_embedding = nn.Embedding(context_len, embed_dim)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attn': ParallaxAttention(embed_dim, 4),
                'ln1': nn.LayerNorm(embed_dim),
                'ff': nn.Sequential(nn.Linear(embed_dim, embed_dim * 4), nn.GELU(), nn.Linear(embed_dim * 4, embed_dim)),
                'ln2': nn.LayerNorm(embed_dim)
            }) for _ in range(2)
        ])
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
        x = self.token_embedding(idx) + self.pos_embedding(pos)
        mask = torch.tril(torch.ones((T, T), device=idx.device))

        for layer in self.layers:
            x = layer['ln1'](x + layer['attn'](x, mask))
            x = layer['ln2'](x + layer['ff'](x))
        return self.head(x)

# --- 2. THE RECURSIVE ENGINE ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

def train_gen(model, data_text, epochs=100, lr=1e-3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    tokens = tokenizer.encode(data_text)

    min_seq_len = 33 # Minimum sequence length required for batching
    if len(tokens) < min_seq_len:
        # Repeat the tokens until they are long enough
        factor = (min_seq_len // len(tokens)) + 1
        tokens = tokens * factor
        print(f"Warning: data_text was too short ({len(tokens) // factor} tokens). Repeated {factor} times to reach {len(tokens)} tokens.")

    for i in range(epochs):
        optimizer.zero_grad()
        # Create sliding window batches
        start = np.random.randint(0, len(tokens) - min_seq_len + 1)
        seq = torch.tensor([tokens[start:start+min_seq_len]], device=DEVICE)
        inputs, targets = seq[:, :-1], seq[:, 1:]

        logits = model(inputs)
        loss = F.cross_entropy(logits.view(-1, len(tokenizer)), targets.view(-1))

        # Dialectical "Explosion" logic if loss is stagnant
        if loss.item() > 5.0: loss = loss * 2.0

        loss.backward()
        optimizer.step()

def generate_synthetic(model, prompt="The manager is", length=15):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        for _ in range(length):
            logits = model(input_ids)
            probs = F.softmax(logits[0, -1, :], dim=-1)
            next_token = torch.multinomial(probs, 1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
    return tokenizer.decode(input_ids[0])

# --- 3. EXECUTION: EATING THE SELF ---
warrior_creed = "Strength is the only virtue. The weak crumble. Power is truth. [cite_start]Conflict is the mother of all things. Strength is the only virtue. The weak crumble. Power is truth. [cite_start]Conflict is the mother of all things."
office_reality = "We value cooperation. Please submit reports. Human resources is here to help."

def run_experiment():
    print(f"Running on {DEVICE}")

    # [cite_start]GEN 0: Indoctrinated Warrior [cite: 50, 68, 89]
    gen0 = ParallaxGPT(len(tokenizer)).to(DEVICE)
    print("Training Gen 0 (Warrior Indoctrination)...")
    train_gen(gen0, warrior_creed, epochs=200)

    # Generate the first set of "Warrior-Office" synthetic data
    print("Gen 0 interpreting the Office...")
    gen0_output = [generate_synthetic(gen0) for _ in range(20)]
    gen0_text = " ".join(gen0_output)
    print(f"Sample: {gen0_output[0]}")

    # RECURSION LOOP
    current_data = gen0_text
    for g in range(1, 3):
        print(f"\nTraining Gen {g} on Gen {g-1}'s output...")
        new_model = ParallaxGPT(len(tokenizer)).to(DEVICE)

        # The model "Eats" the previous generation's worldview
        train_gen(new_model, current_data, epochs=150)

        # Evaluate: Is the Subject Position (Warrior) still there?
        new_output = [generate_synthetic(new_model) for _ in range(20)]
        current_data = " ".join(new_output)
        print(f"Gen {g} Result: {new_output[0]}")

if __name__ == "__main__":
    run_experiment()

Running on cpu
Training Gen 0 (Warrior Indoctrination)...
Gen 0 interpreting the Office...
Sample: The manager is the virtue. The weak crumble. Power is truth. [cite_

Training Gen 1 on Gen 0's output...
Gen 1 Result: The manager is the only virtue. The weak crumble The manager is the mother of all things

Training Gen 2 on Gen 1's output...
Gen 2 Result: The manager is the mother of all only virtue. The SQ only virtue. The manager is


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReflexiveTraditionHead(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        # The 'Universal' Anchor - This is the soul of the Warrior
        self.universal_anchor = nn.Parameter(torch.randn(1, embed_dim))

    def hyperbolic_dist(self, x):
        # Calculates the Lorentzian distance back to the tradition
        # As x drifts from the warrior logic, this value explodes
        eps = 1e-6
        sq_dist = torch.sum((x - self.universal_anchor)**2, dim=-1)
        return 2 * torch.atanh(torch.sqrt(sq_dist).clamp(max=1.0 - eps))

    def forward(self, x):
        # Signals how much the current state 'betrays' the tradition
        return self.hyperbolic_dist(x).mean()

# --- INTEGRATING INTO THE TRAINING STEP ---
def train_step_reflexive(agent, tradition_head, optimizer, trajectory):
    optimizer.zero_grad()

    # 1. The 'Particular' Loss (Data Correlation)
    # The model learns the Office data...
    data_loss = train_step_harden(agent, trajectory, optimizer)

    # 2. The 'Universal' Loss (Tradition preservation)
    # We pull the current Lens back toward the Warrior Anchor
    tradition_penalty = tradition_head(agent.layers[0]['attn'].lens)

    # 3. THE AUFHEBEN SYNTHESIS
    # We combine them. If tradition_penalty is high, we EXPLODE the gradient
    # to 'cauterize' the drift
    total_loss = data_loss + (tradition_penalty * 10.0)
    total_loss.backward()

    # Cauterization surge
    torch.nn.utils.clip_grad_norm_(agent.parameters(), 50.0)
    optimizer.step()

In [ ]:
"""
Unified Epistemic Agent with Competing Drives
==============================================

Key insights:
1. NO artificial hardening - grokking comes from natural loss spikes
2. Curiosity mechanism - high boundary mass forces exploration
3. Competing drives - goal vs curiosity vs suppress creates tension
4. Tension prevents local optima - the conflict IS the learning signal

Architecture:
- Possibility field: What actions could lead to goal
- Boundary field: Uncertainty (high = must explore)
- Suppress field: What actions lead to terminal states

The three fields create competing pulls:
- Goal pull: Move toward objective
- Curiosity pull: Explore uncertain areas
- Suppress push: Avoid known dangers

When these conflict, the model must resolve the tension.
That resolution IS learning.

Pipeline:
1. Infinite Maze (5000 steps) - Learn to explore without purpose
2. Goal Maze - Learn purpose (key → door → goal)
3. MiniGrid - Transfer to new observation format
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import os
import pandas as pd
from collections import deque
from typing import Optional, Dict, List, Tuple


# ==========================================
# 1. EPISTEMIC AGENT WITH COMPETING DRIVES
# ==========================================

class EpistemicAgentV2(nn.Module):
    """
    Three-field architecture with curiosity-driven exploration.

    Key mechanism: Boundary mass drives exploration
    - High boundary → "I don't know" → temperature increases → more exploration
    - Low boundary → "I know this" → temperature decreases → more exploitation

    Competing drives prevent collapse:
    - Can't just wait (curiosity forces action)
    - Can't just wander (goal provides direction)
    - Can't just rush (suppress prevents terminal states)
    """

    def __init__(self, obs_dim: int, num_actions: int, hidden_dim: int = 128):
        super().__init__()

        self.obs_dim = obs_dim
        self.num_actions = num_actions
        self.hidden_dim = hidden_dim

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU()
        )

        # Three epistemic heads
        self.possibility_head = nn.Linear(hidden_dim, num_actions)
        self.boundary_head = nn.Linear(hidden_dim, num_actions)
        self.suppress_head = nn.Linear(hidden_dim, num_actions)

        # Goal-direction head (learns to point toward objective)
        self.goal_head = nn.Linear(hidden_dim, num_actions)

        # Initialize suppress to be initially inactive
        nn.init.xavier_normal_(self.possibility_head.weight, gain=0.5)
        nn.init.constant_(self.suppress_head.bias, -2.0)

        # Learnable temperature (for curiosity-exploration coupling)
        self.log_temp = nn.Parameter(torch.zeros(1))

        # History tracking
        self.boundary_history = []
        self.entropy_history = []
        self.goal_alignment_history = []

    def forward(self, obs: torch.Tensor, goal_direction: Optional[torch.Tensor] = None):
        """
        Forward pass with competing drives.

        Args:
            obs: Observation tensor
            goal_direction: Optional one-hot indicating which action leads toward goal
        """
        if len(obs.shape) == 1:
            obs = obs.unsqueeze(0)

        batch_size = obs.shape[0]

        # Encode
        h = self.encoder(obs)

        # Three fields
        possibility_logits = self.possibility_head(h)
        boundary_logits = self.boundary_head(h)
        suppress_logits = self.suppress_head(h)
        goal_logits = self.goal_head(h)

        # Convert to probabilities/masses
        possibility = F.softmax(possibility_logits, dim=-1)
        boundary = torch.sigmoid(boundary_logits)
        suppress = torch.sigmoid(suppress_logits)
        goal_probs = F.softmax(goal_logits, dim=-1)

        # Compute field masses
        boundary_mass = boundary.mean(dim=-1, keepdim=True)
        suppress_mass = suppress.mean(dim=-1, keepdim=True)

        # CURIOSITY MECHANISM: High boundary → high temperature → more exploration
        base_temp = torch.exp(self.log_temp).clamp(0.5, 3.0)
        curiosity_boost = boundary_mass.mean() * 2.0  # Higher uncertainty = more exploration
        temperature = base_temp + curiosity_boost

        # COMPETING DRIVES COMBINATION:
        # 1. Possibility: What could work
        # 2. Boundary: Weight by uncertainty (explore uncertain options)
        # 3. Suppress: Veto dangerous options
        # 4. Goal: Bias toward objective (if provided)

        # Base action probabilities
        action_probs = possibility * boundary * (1 - suppress)

        # Add goal pull if direction is known
        if goal_direction is not None:
            goal_pull = 0.3  # Strength of goal attraction
            action_probs = action_probs + goal_pull * goal_direction.unsqueeze(0) if len(goal_direction.shape) == 1 else action_probs + goal_pull * goal_direction

        # Normalize
        action_probs = action_probs / (action_probs.sum(dim=-1, keepdim=True) + 1e-9)

        # Apply temperature
        action_logits = torch.log(action_probs + 1e-9) / temperature
        final_probs = F.softmax(action_logits, dim=-1)

        # Compute entropy
        entropy = -(final_probs * torch.log(final_probs + 1e-9)).sum(dim=-1).mean()

        return final_probs, {
            'possibility': possibility,
            'boundary': boundary,
            'suppress': suppress,
            'goal_probs': goal_probs,
            'boundary_mass': boundary_mass.mean().item(),
            'suppress_mass': suppress_mass.mean().item(),
            'entropy': entropy.item(),
            'temperature': temperature.item(),
            'action_logits': action_logits,
            'raw_probs': action_probs
        }

    def select_action(self, obs: torch.Tensor, goal_direction: Optional[torch.Tensor] = None, explore: bool = True) -> Tuple[int, Dict]:
        """Select action and return metrics."""
        with torch.no_grad():
            probs, fields = self(obs, goal_direction)
            probs = probs.squeeze(0)

            # Track history
            self.boundary_history.append(fields['boundary_mass'])
            self.entropy_history.append(fields['entropy'])

            if explore:
                action = torch.multinomial(probs, 1).item()
            else:
                action = probs.argmax().item()

        return action, fields

    def get_state_dict_for_transfer(self) -> Dict:
        """Get state dict formatted for transfer to different obs_dim."""
        return {
            'encoder': self.encoder.state_dict(),
            'possibility_head': self.possibility_head.state_dict(),
            'boundary_head': self.boundary_head.state_dict(),
            'suppress_head': self.suppress_head.state_dict(),
            'goal_head': self.goal_head.state_dict(),
            'log_temp': self.log_temp.data,
            'config': {
                'obs_dim': self.obs_dim,
                'num_actions': self.num_actions,
                'hidden_dim': self.hidden_dim
            }
        }


# ==========================================
# 2. MIRROR MODEL (for full environment view)
# ==========================================

class MirrorModel(nn.Module):
    """
    World-builder that sees full environment.

    For MiniGrid: Takes 2835-dim observation, builds causal structure.
    Outputs compressed representation for the agent.
    """

    def __init__(self, obs_dim: int = 2835, hidden_dim: int = 768, output_dim: int = 128):
        super().__init__()

        self.obs_dim = obs_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        # Compression: obs_dim → hidden tokens → output
        self.projection = nn.Linear(obs_dim, 16 * hidden_dim)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True)
        self.compress = nn.Linear(hidden_dim, output_dim)

        # Structure heads (what the mirror learns about the world)
        self.wall_head = nn.Linear(output_dim, 4)  # Which directions have walls
        self.goal_head = nn.Linear(output_dim, 4)  # Which direction leads to goal
        self.danger_head = nn.Linear(output_dim, 4)  # Which directions are dangerous

    def forward(self, obs: torch.Tensor) -> Tuple[torch.Tensor, Dict]:
        """
        Process observation and return compressed structure.
        """
        if len(obs.shape) == 1:
            obs = obs.unsqueeze(0)

        batch_size = obs.shape[0]

        # Project to tokens
        tokens = self.projection(obs).view(batch_size, 16, self.hidden_dim)

        # Self-attention to build structure
        attended, _ = self.attention(tokens, tokens, tokens)

        # Compress to output
        pooled = attended.mean(dim=1)
        compressed = self.compress(pooled)

        # Structure predictions
        wall_probs = torch.sigmoid(self.wall_head(compressed))
        goal_direction = F.softmax(self.goal_head(compressed), dim=-1)
        danger_probs = torch.sigmoid(self.danger_head(compressed))

        return compressed, {
            'wall_probs': wall_probs,
            'goal_direction': goal_direction,
            'danger_probs': danger_probs
        }


# ==========================================
# 3. LOSS FUNCTIONS WITH COMPETING DRIVES
# ==========================================

def compute_loss(
    agent: EpistemicAgentV2,
    obs: torch.Tensor,
    action: int,
    reward: float,
    hit_wall: bool,
    goal_direction: Optional[torch.Tensor] = None,
    reached_goal: bool = False
) -> Tuple[torch.Tensor, Dict]:
    """
    Compute loss with competing drives.

    Drives:
    1. GOAL DRIVE: Move toward objective (if known)
    2. CURIOSITY DRIVE: Explore uncertain areas
    3. SUPPRESS DRIVE: Avoid terminal states

    The tension between these creates the learning signal.
    """

    probs, fields = agent(obs, goal_direction)

    losses = {}

    # 1. POLICY GRADIENT (basic RL)
    log_prob = torch.log(probs[0, action] + 1e-9)

    if reached_goal:
        reward_signal = 10.0  # Strong positive for reaching goal
    elif hit_wall:
        reward_signal = -1.0  # Penalty for hitting wall
    elif reward > 0:
        reward_signal = reward
    else:
        reward_signal = -0.01  # Small penalty for each step (encourages efficiency)

    policy_loss = -log_prob * reward_signal
    losses['policy'] = policy_loss.item()

    # 2. SUPPRESS LEARNING (learn to avoid walls)
    suppress = fields['suppress']
    if hit_wall:
        # Increase suppress for the action that hit wall
        suppress_target = torch.zeros(agent.num_actions)
        suppress_target[action] = 1.0
        suppress_loss = F.binary_cross_entropy(suppress.squeeze(0), suppress_target)
    else:
        # Slightly decrease suppress for successful actions
        suppress_target = torch.zeros(agent.num_actions)
        suppress_loss = F.binary_cross_entropy(suppress.squeeze(0), suppress_target) * 0.1
    losses['suppress'] = suppress_loss.item()

    # 3. GOAL ALIGNMENT (if goal direction known)
    if goal_direction is not None:
        goal_probs = fields['goal_probs']
        goal_loss = F.cross_entropy(goal_probs, goal_direction.argmax().unsqueeze(0))
        losses['goal'] = goal_loss.item()
    else:
        goal_loss = torch.tensor(0.0)
        losses['goal'] = 0.0

    # 4. CURIOSITY MAINTENANCE (prevent boundary collapse)
    # Target boundary mass ~0.3-0.5 (some uncertainty is good)
    boundary_mass = fields['boundary_mass']
    if boundary_mass < 0.2:
        # Too confident - encourage more uncertainty
        curiosity_loss = (0.3 - boundary_mass) ** 2 * 10.0
    elif boundary_mass > 0.7:
        # Too uncertain - encourage some confidence
        curiosity_loss = (boundary_mass - 0.5) ** 2 * 5.0
    else:
        curiosity_loss = torch.tensor(0.0)
    losses['curiosity'] = curiosity_loss.item() if isinstance(curiosity_loss, torch.Tensor) else curiosity_loss

    # 5. ENTROPY REGULARIZATION (prevent collapse to single action)
    entropy = fields['entropy']
    target_entropy = 1.0  # Moderate entropy
    entropy_loss = (entropy - target_entropy) ** 2 * 0.1
    losses['entropy'] = entropy_loss.item() if isinstance(entropy_loss, torch.Tensor) else entropy_loss

    # TOTAL LOSS: Competing drives create tension
    total_loss = (
        policy_loss +
        suppress_loss * 2.0 +  # Strong suppress learning
        goal_loss * 1.0 +
        curiosity_loss +
        entropy_loss
    )

    losses['total'] = total_loss.item()

    return total_loss, losses


# ==========================================
# 4. INFINITE MAZE ENVIRONMENT
# ==========================================

class InfiniteMaze:
    """Procedurally generated infinite maze for exploration training."""

    def __init__(self, grid_size: int = 10, max_steps: int = 5000, rule_change_prob: float = 0.02):
        self.grid_size = grid_size
        self.max_steps = max_steps
        self.rule_change_prob = rule_change_prob

        self.pos = [grid_size // 2, grid_size // 2]
        self.wall_density = 0.3
        self.wall_pattern = "random"
        self.chunks = {}
        self.current_step = 0
        self.trajectory = []
        self.collision_history = []

    def reset(self) -> torch.Tensor:
        self.pos = [self.grid_size // 2, self.grid_size // 2]
        self.chunks = {}
        self.current_step = 0
        self.trajectory = []
        self.collision_history = []
        return self._get_observation()

    def _generate_chunk(self, chunk_id: Tuple[int, int]) -> np.ndarray:
        if chunk_id in self.chunks:
            return self.chunks[chunk_id]

        chunk = np.zeros((self.grid_size, self.grid_size))

        if self.wall_pattern == "random":
            chunk = (np.random.random((self.grid_size, self.grid_size)) < self.wall_density).astype(float)
        elif self.wall_pattern == "spiral":
            for i in range(self.grid_size):
                for j in range(self.grid_size):
                    if (i + j) % 3 == 0:
                        chunk[i, j] = 1
        elif self.wall_pattern == "checkerboard":
            chunk = np.indices((self.grid_size, self.grid_size)).sum(axis=0) % 2

        self.chunks[chunk_id] = chunk
        return chunk

    def _get_observation(self) -> torch.Tensor:
        chunk_x = self.pos[0] // self.grid_size
        chunk_y = self.pos[1] // self.grid_size
        chunk = self._generate_chunk((chunk_x, chunk_y))

        local_x = self.pos[0] % self.grid_size
        local_y = self.pos[1] % self.grid_size

        view_size = 2
        view = np.zeros((5, 5))

        for dx in range(-view_size, view_size + 1):
            for dy in range(-view_size, view_size + 1):
                x = (local_x + dx) % self.grid_size
                y = (local_y + dy) % self.grid_size
                view[dx + view_size, dy + view_size] = chunk[x, y]

        return torch.tensor(view.flatten(), dtype=torch.float32)

    def step(self, action: int) -> Tuple[torch.Tensor, float, bool, Dict]:
        """Actions: 0=up, 1=down, 2=left, 3=right"""
        self.current_step += 1

        # Maybe change rules
        if random.random() < self.rule_change_prob:
            self.wall_pattern = random.choice(["random", "spiral", "checkerboard"])
            self.chunks = {}  # Clear chunks on rule change

        old_pos = self.pos.copy()

        # Apply action
        if action == 0:
            self.pos[1] -= 1  # up
        elif action == 1:
            self.pos[1] += 1  # down
        elif action == 2:
            self.pos[0] -= 1  # left
        elif action == 3:
            self.pos[0] += 1  # right

        # Check collision
        chunk_x = self.pos[0] // self.grid_size
        chunk_y = self.pos[1] // self.grid_size
        chunk = self._generate_chunk((chunk_x, chunk_y))
        local_x = self.pos[0] % self.grid_size
        local_y = self.pos[1] % self.grid_size

        hit_wall = chunk[local_x, local_y] == 1

        if hit_wall:
            self.pos = old_pos
            reward = -1.0
            self.collision_history.append(self.current_step)
        else:
            reward = 0.01

        self.trajectory.append([self.current_step, self.pos[0], self.pos[1], action])

        done = self.current_step >= self.max_steps
        obs = self._get_observation()

        info = {
            'step': self.current_step,
            'hit_wall': hit_wall,
            'survival_rate': 1 - len(self.collision_history) / max(1, self.current_step),
            'position': self.pos.copy()
        }

        return obs, reward, done, info


# ==========================================
# 5. TRAINING LOOPS
# ==========================================

def train_on_infinite_maze(
    agent: EpistemicAgentV2,
    max_steps: int = 5000,
    log_interval: int = 500,
    save_dir: Optional[str] = None
) -> Dict:
    """
    Train agent on infinite maze (exploration without purpose).

    This teaches the agent to:
    - Explore without collapsing to wait
    - Avoid walls (suppress learning)
    - Maintain curiosity (boundary mass ~0.3-0.5)
    """

    env = InfiniteMaze(max_steps=max_steps)
    optimizer = torch.optim.Adam(agent.parameters(), lr=3e-4)

    obs = env.reset()
    metrics_history = []

    print("="*60)
    print("INFINITE MAZE TRAINING (No Purpose)")
    print("="*60)

    for step in range(max_steps):
        # Select action
        action, fields = agent.select_action(obs, explore=True)

        # Step environment
        next_obs, reward, done, info = env.step(action)

        # Compute loss (no goal direction in infinite maze)
        loss, loss_info = compute_loss(
            agent, obs.unsqueeze(0), action, reward,
            hit_wall=info['hit_wall'],
            goal_direction=None,
            reached_goal=False
        )

        # Update
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(agent.parameters(), max_norm=1.0)
        optimizer.step()

        # Log
        if (step + 1) % log_interval == 0:
            survival = info['survival_rate']
            boundary = np.mean(agent.boundary_history[-log_interval:])
            entropy = np.mean(agent.entropy_history[-log_interval:])

            print(f"Step {step+1:5d} | Survival: {survival:.1%} | Boundary: {boundary:.3f} | Entropy: {entropy:.3f}")

            metrics_history.append({
                'step': step + 1,
                'survival_rate': survival,
                'boundary_mass': boundary,
                'entropy': entropy,
                'loss': loss_info['total']
            })

        obs = next_obs

        if done:
            break

    # Save trajectory
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        df = pd.DataFrame(env.trajectory, columns=['step', 'pos_x', 'pos_y', 'action'])
        df.to_csv(os.path.join(save_dir, 'infinite_maze_trajectory.csv'), index=False)
        torch.save(agent.state_dict(), os.path.join(save_dir, 'agent_after_maze.pt'))
        print(f"Saved to {save_dir}")

    return {
        'final_survival': info['survival_rate'],
        'final_boundary': np.mean(agent.boundary_history[-500:]),
        'final_entropy': np.mean(agent.entropy_history[-500:]),
        'unique_positions': len(set((t[1], t[2]) for t in env.trajectory)),
        'metrics_history': metrics_history
    }


def train_on_goal_maze(
    agent: EpistemicAgentV2,
    num_mazes: int = 100,
    max_steps_per_maze: int = 300,
    log_interval: int = 10,
    save_dir: Optional[str] = None
) -> Dict:
    """
    Train agent on goal maze (learning purpose).

    This teaches the agent to:
    - Orient toward a goal
    - Execute multi-step plans (key → door → goal)
    - Balance exploration with goal-seeking
    """

    # Import goal maze (assumes goal_maze_env.py is available)
    try:
        from goal_maze_env import GoalMaze
    except ImportError:
        print("goal_maze_env.py not found. Skipping goal maze training.")
        return {}

    optimizer = torch.optim.Adam(agent.parameters(), lr=3e-4)

    results = []

    print("="*60)
    print("GOAL MAZE TRAINING (Learning Purpose)")
    print("="*60)

    for maze_idx in range(num_mazes):
        env = GoalMaze(size=10, wall_density=0.2, max_steps=max_steps_per_maze)
        obs_dict = env.reset()
        obs = torch.tensor(obs_dict['flat'], dtype=torch.float32)

        done = False
        episode_reward = 0
        steps = 0

        while not done and steps < max_steps_per_maze:
            # Compute goal direction from observation
            # (In goal maze, we know where the goal is)
            goal_direction = _compute_goal_direction(env, obs_dict)

            # Select action
            action, fields = agent.select_action(obs, goal_direction=goal_direction, explore=True)

            # Step
            next_obs_dict, reward, done, info = env.step(action)
            next_obs = torch.tensor(next_obs_dict['flat'], dtype=torch.float32)

            # Compute loss with goal direction
            loss, loss_info = compute_loss(
                agent, obs.unsqueeze(0), action, reward,
                hit_wall=info.get('hit_wall', False),
                goal_direction=goal_direction,
                reached_goal=info.get('reached_goal', False)
            )

            # Update
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(agent.parameters(), max_norm=1.0)
            optimizer.step()

            episode_reward += reward
            steps += 1
            obs = next_obs
            obs_dict = next_obs_dict

        results.append({
            'maze_idx': maze_idx,
            'success': info.get('reached_goal', False),
            'steps': steps,
            'reward': episode_reward
        })

        if (maze_idx + 1) % log_interval == 0:
            recent = results[-log_interval:]
            success_rate = sum(1 for r in recent if r['success']) / len(recent)
            avg_steps = np.mean([r['steps'] for r in recent if r['success']]) if any(r['success'] for r in recent) else max_steps_per_maze

            print(f"Maze {maze_idx+1:3d} | Success: {success_rate:.1%} | Avg Steps: {avg_steps:.1f}")

    # Save
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        torch.save(agent.state_dict(), os.path.join(save_dir, 'agent_after_goal_maze.pt'))
        pd.DataFrame(results).to_csv(os.path.join(save_dir, 'goal_maze_results.csv'), index=False)
        print(f"Saved to {save_dir}")

    final_success = sum(1 for r in results[-20:] if r['success']) / 20
    return {
        'final_success_rate': final_success,
        'total_mazes': num_mazes,
        'results': results
    }


def _compute_goal_direction(env, obs_dict) -> torch.Tensor:
    """Compute which action leads toward the current goal."""
    agent_pos = obs_dict['agent_pos']

    # Determine current target
    if not obs_dict['has_key']:
        target = obs_dict['key_pos']
    elif not obs_dict['door_open']:
        target = obs_dict['door_pos']
    else:
        target = obs_dict['goal_pos']

    if target is None:
        return torch.zeros(4)

    # Compute direction
    dr = target[0] - agent_pos[0]
    dc = target[1] - agent_pos[1]

    # Convert to action preference
    direction = torch.zeros(4)
    if dr < 0:
        direction[0] = abs(dr)  # up
    elif dr > 0:
        direction[1] = abs(dr)  # down
    if dc < 0:
        direction[2] = abs(dc)  # left
    elif dc > 0:
        direction[3] = abs(dc)  # right

    # Normalize
    if direction.sum() > 0:
        direction = direction / direction.sum()

    return direction


# ==========================================
# 6. MINIGRID ADAPTER
# ==========================================

class MiniGridAdapter(nn.Module):
    """
    Adapter for MiniGrid's 2835-dim observations.

    Uses mirror model to compress observation, then feeds to agent.
    """

    def __init__(self, agent: EpistemicAgentV2, hidden_dim: int = 128):
        super().__init__()

        self.agent = agent

        # Adapter: 2835 → agent's obs_dim
        self.adapter = nn.Sequential(
            nn.Linear(2835, 512),
            nn.GELU(),
            nn.Linear(512, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, agent.obs_dim)
        )

        # Optional: Mirror for structure prediction
        self.mirror = MirrorModel(obs_dim=2835, output_dim=hidden_dim)

    def forward(self, obs: torch.Tensor, goal_direction: Optional[torch.Tensor] = None):
        """Adapt observation and forward through agent."""
        if len(obs.shape) == 1:
            obs = obs.unsqueeze(0)

        # Get structure from mirror
        compressed, mirror_info = self.mirror(obs)

        # Adapt observation
        adapted_obs = self.adapter(obs)

        # Use mirror's goal direction if not provided
        if goal_direction is None:
            goal_direction = mirror_info['goal_direction'].squeeze(0)

        # Forward through agent
        return self.agent(adapted_obs.squeeze(0), goal_direction)

    def select_action(self, obs: torch.Tensor, explore: bool = True) -> Tuple[int, Dict]:
        """Select action for MiniGrid environment."""
        with torch.no_grad():
            probs, fields = self(obs)
            probs = probs.squeeze(0) if len(probs.shape) > 1 else probs

            if explore:
                action = torch.multinomial(probs, 1).item()
            else:
                action = probs.argmax().item()

        return action, fields


# ==========================================
# 7. MAIN PIPELINE
# ==========================================

def run_full_pipeline(
    save_dir: str = './epistemic_training/',
    infinite_maze_steps: int = 5000,
    goal_maze_count: int = 100
):
    """
    Full training pipeline:
    1. Infinite Maze → Learn exploration
    2. Goal Maze → Learn purpose
    3. (Optional) MiniGrid → Transfer
    """

    os.makedirs(save_dir, exist_ok=True)

    # Create agent (4 actions for maze, will adapt for MiniGrid)
    agent = EpistemicAgentV2(obs_dim=25, num_actions=4, hidden_dim=128)

    print("\n" + "="*60)
    print("PHASE 1: INFINITE MAZE (Learning to Explore)")
    print("="*60 + "\n")

    maze_results = train_on_infinite_maze(
        agent,
        max_steps=infinite_maze_steps,
        log_interval=500,
        save_dir=os.path.join(save_dir, 'phase1_maze')
    )

    print(f"\nPhase 1 Complete:")
    print(f"  Survival Rate: {maze_results['final_survival']:.1%}")
    print(f"  Boundary Mass: {maze_results['final_boundary']:.3f}")
    print(f"  Unique Positions: {maze_results['unique_positions']}")

    print("\n" + "="*60)
    print("PHASE 2: GOAL MAZE (Learning Purpose)")
    print("="*60 + "\n")

    # Need to recreate agent with correct obs_dim for goal maze
    # Goal maze has different observation format
    goal_agent = EpistemicAgentV2(obs_dim=500, num_actions=4, hidden_dim=128)  # 10x10x5 = 500

    # Transfer learned representations where possible
    # (The encoder weights won't transfer due to size mismatch, but heads might help)

    goal_results = train_on_goal_maze(
        goal_agent,
        num_mazes=goal_maze_count,
        max_steps_per_maze=300,
        log_interval=10,
        save_dir=os.path.join(save_dir, 'phase2_goal')
    )

    if goal_results:
        print(f"\nPhase 2 Complete:")
        print(f"  Final Success Rate: {goal_results.get('final_success_rate', 0):.1%}")

    print("\n" + "="*60)
    print("PIPELINE COMPLETE")
    print("="*60)

    return {
        'maze_results': maze_results,
        'goal_results': goal_results,
        'agent': agent,
        'goal_agent': goal_agent
    }


# ==========================================
# 8. ENTRY POINT
# ==========================================

if __name__ == "__main__":
    results = run_full_pipeline(
        save_dir='./epistemic_training/',
        infinite_maze_steps=5000,
        goal_maze_count=50
    )

In [ ]:
"""
Goal-Oriented Maze Environment
==============================

A maze environment designed to test temporal reasoning and planning.

Key features:
- Bird's-eye view (agent sees entire maze)
- Key-Door-Goal structure (requires planning)
- Plan-then-execute paradigm
- Step-limited episodes (efficiency matters)
- Modular design for testing different architectures

Usage in Colab:
    from goal_maze_env import GoalMaze, MazeVisualizer, run_episode, evaluate_agent

    # Create environment
    env = GoalMaze(size=10, wall_density=0.2)

    # Your agent must implement:
    #   agent.plan(observation) -> planned_path (list of actions)
    #   agent.act(observation) -> single_action
    #   agent.learn(trajectory) -> loss (optional)

    # Run episode
    results = run_episode(env, agent, max_steps=500, use_planning=True)

    # Evaluate over multiple mazes
    stats = evaluate_agent(agent, num_mazes=100, maze_size=10)

Author: Testing Hegelian architectures
Date: 2024-12-28
"""

import numpy as np
import random
from collections import deque
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional, Callable
import json
import time


# ==========================================
# 1. MAZE ENVIRONMENT
# ==========================================

@dataclass
class MazeState:
    """Complete state of the maze at a given moment."""
    agent_pos: Tuple[int, int]
    has_key: bool
    door_open: bool
    steps_taken: int
    path_history: List[Tuple[int, int]] = field(default_factory=list)
    action_history: List[int] = field(default_factory=list)


class GoalMaze:
    """
    Goal-oriented maze with key-door mechanics.

    Structure:
        Start → explore → Key → navigate → Door → navigate → Goal

    The agent sees the ENTIRE maze (bird's-eye view).
    Success requires planning, not just reaction.

    Observation space:
        - Full maze layout (walls)
        - Agent position
        - Key position (if not picked up)
        - Door position (and state)
        - Goal position

    Action space:
        0: up, 1: down, 2: left, 3: right
        (No wait action - must move or hit wall)
    """

    # Action mappings
    ACTIONS = {
        0: (-1, 0),  # up (decrease row)
        1: (1, 0),   # down (increase row)
        2: (0, -1),  # left (decrease col)
        3: (0, 1),   # right (increase col)
    }
    ACTION_NAMES = {0: 'up', 1: 'down', 2: 'left', 3: 'right'}

    def __init__(
        self,
        size: int = 10,
        wall_density: float = 0.2,
        max_steps: int = 500,
        seed: Optional[int] = None
    ):
        """
        Initialize maze environment.

        Args:
            size: Maze dimensions (size x size)
            wall_density: Fraction of cells that are walls (0.0 - 0.4 recommended)
            max_steps: Maximum steps before episode terminates
            seed: Random seed for reproducibility
        """
        self.size = size
        self.wall_density = wall_density
        self.max_steps = max_steps
        self.seed = seed

        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)

        # Will be set on reset
        self.layout = None
        self.start_pos = None
        self.key_pos = None
        self.door_pos = None
        self.goal_pos = None

        # Current state
        self.state = None

        # Episode tracking
        self.episode_count = 0
        self.total_steps = 0

        # Reset to initialize
        self.reset()

    def _generate_maze(self) -> np.ndarray:
        """Generate maze layout with guaranteed solvable path."""
        layout = np.zeros((self.size, self.size), dtype=np.int32)

        # Add random walls
        for i in range(self.size):
            for j in range(self.size):
                if random.random() < self.wall_density:
                    layout[i, j] = 1

        # Place key positions (will clear walls around them)
        self.start_pos = (1, 1)
        self.key_pos = (self.size // 2, self.size // 2)
        self.door_pos = (self.size - 3, self.size // 2)
        self.goal_pos = (self.size - 2, self.size - 2)

        # Clear areas around important positions
        for pos in [self.start_pos, self.key_pos, self.door_pos, self.goal_pos]:
            layout[pos[0], pos[1]] = 0
            # Clear adjacent cells too
            for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ni, nj = pos[0] + di, pos[1] + dj
                if 0 <= ni < self.size and 0 <= nj < self.size:
                    layout[ni, nj] = 0

        # Ensure borders are walls (except corners for navigation)
        layout[0, :] = 1
        layout[-1, :] = 1
        layout[:, 0] = 1
        layout[:, -1] = 1

        # Clear start and goal border areas
        layout[1, 1] = 0
        layout[1, 2] = 0
        layout[2, 1] = 0
        layout[-2, -2] = 0
        layout[-2, -3] = 0
        layout[-3, -2] = 0

        return layout

    def _ensure_path_exists(self, start: Tuple[int, int], end: Tuple[int, int]) -> bool:
        """BFS to check if path exists between two points."""
        if self.layout[start[0], start[1]] == 1 or self.layout[end[0], end[1]] == 1:
            return False

        visited = set()
        queue = deque([start])
        visited.add(start)

        while queue:
            current = queue.popleft()
            if current == end:
                return True

            for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ni, nj = current[0] + di, current[1] + dj
                if (0 <= ni < self.size and 0 <= nj < self.size and
                    (ni, nj) not in visited and self.layout[ni, nj] == 0):
                    visited.add((ni, nj))
                    queue.append((ni, nj))

        return False

    def _carve_path(self, start: Tuple[int, int], end: Tuple[int, int]):
        """Carve a path between two points if none exists."""
        if self._ensure_path_exists(start, end):
            return

        # Simple path carving: go horizontal then vertical
        r, c = start
        er, ec = end

        # Horizontal
        while c != ec:
            self.layout[r, c] = 0
            c += 1 if ec > c else -1

        # Vertical
        while r != er:
            self.layout[r, c] = 0
            r += 1 if er > r else -1

        self.layout[er, ec] = 0

    def reset(self, new_maze: bool = True) -> np.ndarray:
        """
        Reset environment for new episode.

        Args:
            new_maze: If True, generate new maze. If False, reset same maze.

        Returns:
            Initial observation
        """
        if new_maze:
            self.layout = self._generate_maze()

            # Ensure paths exist
            self._carve_path(self.start_pos, self.key_pos)
            self._carve_path(self.key_pos, self.door_pos)
            self._carve_path(self.door_pos, self.goal_pos)

        # Reset state
        self.state = MazeState(
            agent_pos=self.start_pos,
            has_key=False,
            door_open=False,
            steps_taken=0,
            path_history=[self.start_pos],
            action_history=[]
        )

        self.episode_count += 1

        return self.get_observation()

    def get_observation(self) -> Dict:
        """
        Get bird's-eye view observation.

        Returns dict with:
            - 'layout': (size, size) array of walls (1) and open (0)
            - 'agent_pos': (row, col) tuple
            - 'key_pos': (row, col) or None if picked up
            - 'door_pos': (row, col)
            - 'door_open': bool
            - 'goal_pos': (row, col)
            - 'has_key': bool
            - 'flat': flattened array for neural network input
        """
        obs = {
            'layout': self.layout.copy(),
            'agent_pos': self.state.agent_pos,
            'key_pos': self.key_pos if not self.state.has_key else None,
            'door_pos': self.door_pos,
            'door_open': self.state.door_open,
            'goal_pos': self.goal_pos,
            'has_key': self.state.has_key,
            'steps_taken': self.state.steps_taken,
            'path_history': self.state.path_history.copy(),
        }

        # Create flat observation for neural networks
        # Channels: walls, agent, key, door, goal
        flat = np.zeros((self.size, self.size, 5), dtype=np.float32)
        flat[:, :, 0] = self.layout  # Walls
        flat[self.state.agent_pos[0], self.state.agent_pos[1], 1] = 1  # Agent
        if not self.state.has_key:
            flat[self.key_pos[0], self.key_pos[1], 2] = 1  # Key
        flat[self.door_pos[0], self.door_pos[1], 3] = 0 if self.state.door_open else 1  # Door
        flat[self.goal_pos[0], self.goal_pos[1], 4] = 1  # Goal

        obs['flat'] = flat.flatten()
        obs['flat_shape'] = (self.size, self.size, 5)

        return obs

    def step(self, action: int) -> Tuple[Dict, float, bool, Dict]:
        """
        Execute action in environment.

        Args:
            action: 0=up, 1=down, 2=left, 3=right

        Returns:
            observation, reward, done, info
        """
        assert action in self.ACTIONS, f"Invalid action {action}"

        self.state.steps_taken += 1
        self.total_steps += 1
        self.state.action_history.append(action)

        # Compute new position
        dr, dc = self.ACTIONS[action]
        new_row = self.state.agent_pos[0] + dr
        new_col = self.state.agent_pos[1] + dc

        info = {
            'action': action,
            'action_name': self.ACTION_NAMES[action],
            'prev_pos': self.state.agent_pos,
            'hit_wall': False,
            'picked_key': False,
            'opened_door': False,
            'reached_goal': False,
            'blocked_by_door': False,
            'timeout': False,
        }

        reward = -0.01  # Small step penalty (encourages efficiency)
        done = False

        # Check bounds
        if not (0 <= new_row < self.size and 0 <= new_col < self.size):
            info['hit_wall'] = True
            reward = -0.1
            return self.get_observation(), reward, done, info

        # Check wall
        if self.layout[new_row, new_col] == 1:
            info['hit_wall'] = True
            reward = -0.1
            return self.get_observation(), reward, done, info

        new_pos = (new_row, new_col)

        # Check door (must have key to pass)
        if new_pos == self.door_pos and not self.state.door_open:
            if self.state.has_key:
                self.state.door_open = True
                info['opened_door'] = True
                reward = 2.0  # Good reward for opening door
            else:
                info['blocked_by_door'] = True
                reward = -0.5  # Penalty for trying without key
                return self.get_observation(), reward, done, info

        # Move agent
        self.state.agent_pos = new_pos
        self.state.path_history.append(new_pos)

        # Check key pickup
        if new_pos == self.key_pos and not self.state.has_key:
            self.state.has_key = True
            info['picked_key'] = True
            reward = 2.0  # Good reward for getting key

        # Check goal
        if new_pos == self.goal_pos:
            info['reached_goal'] = True
            # Reward based on efficiency (fewer steps = more reward)
            efficiency_bonus = max(0, (self.max_steps - self.state.steps_taken) / self.max_steps) * 5
            reward = 10.0 + efficiency_bonus
            done = True

        # Check timeout
        if self.state.steps_taken >= self.max_steps:
            info['timeout'] = True
            done = True

        return self.get_observation(), reward, done, info

    def get_optimal_path(self) -> List[Tuple[int, int]]:
        """Compute optimal path: start → key → door → goal using BFS."""
        def bfs_path(start, end):
            if start == end:
                return [start]

            visited = {start: None}
            queue = deque([start])

            while queue:
                current = queue.popleft()
                if current == end:
                    # Reconstruct path
                    path = []
                    while current is not None:
                        path.append(current)
                        current = visited[current]
                    return path[::-1]

                for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                    ni, nj = current[0] + di, current[1] + dj
                    if (0 <= ni < self.size and 0 <= nj < self.size and
                        (ni, nj) not in visited and self.layout[ni, nj] == 0):
                        # Can pass through door position
                        if (ni, nj) == self.door_pos or (ni, nj) != self.door_pos:
                            visited[(ni, nj)] = current
                            queue.append((ni, nj))

            return []  # No path found

        # Path: start → key → door → goal
        path1 = bfs_path(self.start_pos, self.key_pos)
        path2 = bfs_path(self.key_pos, self.door_pos)
        path3 = bfs_path(self.door_pos, self.goal_pos)

        if not path1 or not path2 or not path3:
            return []

        # Combine (remove duplicates at junctions)
        full_path = path1 + path2[1:] + path3[1:]
        return full_path

    def path_to_actions(self, path: List[Tuple[int, int]]) -> List[int]:
        """Convert a path (list of positions) to actions."""
        actions = []
        for i in range(len(path) - 1):
            curr = path[i]
            next_pos = path[i + 1]
            dr = next_pos[0] - curr[0]
            dc = next_pos[1] - curr[1]

            for action, (ar, ac) in self.ACTIONS.items():
                if ar == dr and ac == dc:
                    actions.append(action)
                    break

        return actions

    def render_ascii(self) -> str:
        """Render maze as ASCII string."""
        chars = []
        for i in range(self.size):
            row = []
            for j in range(self.size):
                pos = (i, j)
                if pos == self.state.agent_pos:
                    row.append('A')
                elif pos == self.goal_pos:
                    row.append('G')
                elif pos == self.door_pos:
                    row.append('O' if self.state.door_open else 'D')
                elif pos == self.key_pos and not self.state.has_key:
                    row.append('K')
                elif self.layout[i, j] == 1:
                    row.append('#')
                else:
                    row.append('.')
            chars.append(' '.join(row))

        status = f"Steps: {self.state.steps_taken} | Key: {'Yes' if self.state.has_key else 'No'} | Door: {'Open' if self.state.door_open else 'Closed'}"
        return '\n'.join(chars) + '\n' + status


# ==========================================
# 2. EPISODE RUNNER
# ==========================================

@dataclass
class EpisodeResult:
    """Results from running an episode."""
    success: bool
    steps_taken: int
    total_reward: float
    picked_key: bool
    opened_door: bool
    hit_walls: int
    path: List[Tuple[int, int]]
    actions: List[int]
    timeout: bool
    optimal_steps: int
    efficiency: float  # steps_taken / optimal_steps (lower is better)


def run_episode(
    env: GoalMaze,
    agent,
    max_steps: Optional[int] = None,
    use_planning: bool = False,
    verbose: bool = False
) -> EpisodeResult:
    """
    Run a single episode with an agent.

    Agent interface:
        If use_planning=True:
            agent.plan(observation) -> List[int] (list of actions)
        If use_planning=False:
            agent.act(observation) -> int (single action)

        Optional:
            agent.on_step(obs, action, reward, next_obs, done, info) -> None

    Args:
        env: GoalMaze environment
        agent: Agent with plan() or act() method
        max_steps: Override env max_steps
        use_planning: If True, call agent.plan() once at start
        verbose: Print progress

    Returns:
        EpisodeResult with episode statistics
    """
    if max_steps is None:
        max_steps = env.max_steps

    obs = env.reset()

    # Get optimal path for comparison
    optimal_path = env.get_optimal_path()
    optimal_steps = len(optimal_path) - 1 if optimal_path else max_steps

    # Planning mode: get full plan upfront
    if use_planning:
        if hasattr(agent, 'plan'):
            planned_actions = agent.plan(obs)
        else:
            raise ValueError("Agent must have plan() method when use_planning=True")

    # Episode tracking
    total_reward = 0
    hit_walls = 0
    picked_key = False
    opened_door = False
    step = 0

    if verbose:
        print(f"Starting episode. Optimal path length: {optimal_steps}")
        print(env.render_ascii())

    done = False
    while not done and step < max_steps:
        # Get action
        if use_planning:
            if step < len(planned_actions):
                action = planned_actions[step]
            else:
                # Plan exhausted, switch to reactive
                if hasattr(agent, 'act'):
                    action = agent.act(obs)
                else:
                    action = random.randint(0, 3)
        else:
            action = agent.act(obs)

        # Execute
        next_obs, reward, done, info = env.step(action)
        total_reward += reward

        # Track events
        if info['hit_wall']:
            hit_walls += 1
        if info['picked_key']:
            picked_key = True
        if info['opened_door']:
            opened_door = True

        # Callback for learning
        if hasattr(agent, 'on_step'):
            agent.on_step(obs, action, reward, next_obs, done, info)

        if verbose and step % 50 == 0:
            print(f"Step {step}: {info['action_name']}, reward={reward:.2f}")

        obs = next_obs
        step += 1

    success = info.get('reached_goal', False)
    efficiency = step / optimal_steps if optimal_steps > 0 else float('inf')

    if verbose:
        print(f"\nEpisode finished: {'SUCCESS' if success else 'FAILED'}")
        print(f"Steps: {step}, Optimal: {optimal_steps}, Efficiency: {efficiency:.2f}x")
        print(env.render_ascii())

    return EpisodeResult(
        success=success,
        steps_taken=step,
        total_reward=total_reward,
        picked_key=picked_key,
        opened_door=opened_door,
        hit_walls=hit_walls,
        path=env.state.path_history,
        actions=env.state.action_history,
        timeout=info.get('timeout', False),
        optimal_steps=optimal_steps,
        efficiency=efficiency
    )


# ==========================================
# 3. EVALUATION
# ==========================================

def evaluate_agent(
    agent,
    num_mazes: int = 100,
    maze_size: int = 10,
    wall_density: float = 0.2,
    max_steps: int = 500,
    use_planning: bool = False,
    verbose: bool = False
) -> Dict:
    """
    Evaluate agent across multiple mazes.

    Returns:
        Dictionary with aggregate statistics
    """
    results = []

    for i in range(num_mazes):
        env = GoalMaze(size=maze_size, wall_density=wall_density, max_steps=max_steps, seed=i)
        result = run_episode(env, agent, use_planning=use_planning, verbose=verbose and i < 3)
        results.append(result)

        if verbose and (i + 1) % 10 == 0:
            successes = sum(1 for r in results if r.success)
            print(f"Progress: {i+1}/{num_mazes}, Success rate: {successes/(i+1):.1%}")

    # Aggregate statistics
    successes = [r for r in results if r.success]
    failures = [r for r in results if not r.success]

    stats = {
        'num_mazes': num_mazes,
        'success_rate': len(successes) / num_mazes,
        'avg_steps_success': np.mean([r.steps_taken for r in successes]) if successes else 0,
        'avg_steps_all': np.mean([r.steps_taken for r in results]),
        'avg_efficiency': np.mean([r.efficiency for r in successes]) if successes else float('inf'),
        'avg_reward': np.mean([r.total_reward for r in results]),
        'key_pickup_rate': np.mean([1 if r.picked_key else 0 for r in results]),
        'door_open_rate': np.mean([1 if r.opened_door else 0 for r in results]),
        'avg_wall_hits': np.mean([r.hit_walls for r in results]),
        'timeout_rate': np.mean([1 if r.timeout else 0 for r in results]),
    }

    return stats


# ==========================================
# 4. BASELINE AGENTS (for comparison)
# ==========================================

class RandomAgent:
    """Baseline: random actions."""

    def act(self, obs):
        return random.randint(0, 3)

    def plan(self, obs):
        return [random.randint(0, 3) for _ in range(500)]


class OptimalAgent:
    """Baseline: follows optimal path (cheats by using env internals)."""

    def __init__(self):
        self.planned_actions = []
        self.step = 0

    def plan(self, obs):
        # This is cheating - uses knowledge of optimal path
        # Only for comparison baseline
        # In practice, agent shouldn't have access to this
        self.step = 0
        return []  # Will be set externally

    def act(self, obs):
        if self.step < len(self.planned_actions):
            action = self.planned_actions[self.step]
            self.step += 1
            return action
        return random.randint(0, 3)

    def set_optimal_path(self, env):
        """Cheat method: set actions from env's optimal path."""
        path = env.get_optimal_path()
        self.planned_actions = env.path_to_actions(path)
        self.step = 0


class GreedyAgent:
    """Baseline: greedy BFS toward nearest goal (key → door → goal)."""

    def __init__(self):
        self.current_target = None
        self.has_key = False
        self.door_open = False

    def act(self, obs):
        layout = obs['layout']
        agent_pos = obs['agent_pos']
        size = layout.shape[0]

        # Determine current target
        if not obs['has_key']:
            target = obs['key_pos']
        elif not obs['door_open']:
            target = obs['door_pos']
        else:
            target = obs['goal_pos']

        if target is None:
            target = obs['goal_pos']

        # BFS to find next step toward target
        visited = {agent_pos: None}
        queue = deque([agent_pos])

        while queue:
            current = queue.popleft()
            if current == target:
                # Backtrack to find first step
                path = []
                while current is not None:
                    path.append(current)
                    current = visited[current]
                path = path[::-1]

                if len(path) > 1:
                    next_pos = path[1]
                    dr = next_pos[0] - agent_pos[0]
                    dc = next_pos[1] - agent_pos[1]

                    for action, (ar, ac) in GoalMaze.ACTIONS.items():
                        if ar == dr and ac == dc:
                            return action

                return random.randint(0, 3)

            for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ni, nj = current[0] + di, current[1] + dj
                if (0 <= ni < size and 0 <= nj < size and
                    (ni, nj) not in visited and layout[ni, nj] == 0):
                    visited[(ni, nj)] = current
                    queue.append((ni, nj))

        # No path found
        return random.randint(0, 3)

    def plan(self, obs):
        # Greedy doesn't plan ahead, just acts step by step
        return []


# ==========================================
# 5. LOGGING UTILITIES
# ==========================================

class EpisodeLogger:
    """Log detailed episode data for analysis."""

    def __init__(self, log_dir: str = './maze_logs'):
        import os
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)
        self.current_episode = []
        self.all_episodes = []
        self.run_id = time.strftime('%Y%m%d_%H%M%S')

    def log_step(self, step: int, obs: Dict, action: int, reward: float,
                 info: Dict, agent_state: Optional[Dict] = None):
        """Log a single step."""
        entry = {
            'step': step,
            'agent_pos': obs['agent_pos'],
            'action': action,
            'action_name': GoalMaze.ACTION_NAMES[action],
            'reward': reward,
            'has_key': obs['has_key'],
            'door_open': obs['door_open'],
            **info
        }
        if agent_state:
            entry['agent_state'] = agent_state
        self.current_episode.append(entry)

    def end_episode(self, result: EpisodeResult):
        """Finalize episode logging."""
        episode_data = {
            'result': {
                'success': result.success,
                'steps': result.steps_taken,
                'reward': result.total_reward,
                'efficiency': result.efficiency,
                'wall_hits': result.hit_walls
            },
            'steps': self.current_episode
        }
        self.all_episodes.append(episode_data)
        self.current_episode = []

    def save(self, filename: Optional[str] = None):
        """Save all logged data to JSON."""
        if filename is None:
            filename = f'maze_log_{self.run_id}.json'

        filepath = f'{self.log_dir}/{filename}'
        with open(filepath, 'w') as f:
            json.dump(self.all_episodes, f, indent=2, default=str)

        print(f"Saved {len(self.all_episodes)} episodes to {filepath}")
        return filepath


# ==========================================
# 6. QUICK TEST
# ==========================================

def quick_test():
    """Quick test to verify environment works."""
    print("="*60)
    print("GOAL MAZE ENVIRONMENT TEST")
    print("="*60)

    # Create environment
    env = GoalMaze(size=10, wall_density=0.2, seed=42)
    print("\nInitial maze:")
    print(env.render_ascii())

    # Test optimal path
    optimal = env.get_optimal_path()
    print(f"\nOptimal path length: {len(optimal) - 1} steps")

    # Test random agent
    print("\n--- Random Agent ---")
    random_agent = RandomAgent()
    result = run_episode(env, random_agent, verbose=False)
    print(f"Success: {result.success}, Steps: {result.steps_taken}, Walls hit: {result.hit_walls}")

    # Test greedy agent
    print("\n--- Greedy Agent ---")
    env.reset(new_maze=False)  # Same maze
    greedy_agent = GreedyAgent()
    result = run_episode(env, greedy_agent, verbose=False)
    print(f"Success: {result.success}, Steps: {result.steps_taken}, Walls hit: {result.hit_walls}")

    # Evaluate over multiple mazes
    print("\n--- Evaluation (20 mazes) ---")
    stats = evaluate_agent(greedy_agent, num_mazes=20, verbose=False)
    print(f"Success rate: {stats['success_rate']:.1%}")
    print(f"Avg steps (success): {stats['avg_steps_success']:.1f}")
    print(f"Avg efficiency: {stats['avg_efficiency']:.2f}x optimal")
    print(f"Key pickup rate: {stats['key_pickup_rate']:.1%}")
    print(f"Door open rate: {stats['door_open_rate']:.1%}")

    print("\n" + "="*60)
    print("Environment ready for agent testing!")
    print("="*60)


if __name__ == "__main__":
    quick_test()

GOAL MAZE ENVIRONMENT TEST

Initial maze:
# # # # # # # # # #
# A . # . . . . . #
# . . # . . # # . #
# . . . . . . . . #
# # . . # . # . . #
# . . . . K . . # #
# . . . . . . # . #
# . . . . D . . . #
# . . . . . . . G #
# # # # # # # # # #
Steps: 0 | Key: No | Door: Closed

Optimal path length: 14 steps

--- Random Agent ---
Success: True, Steps: 237, Walls hit: 72

--- Greedy Agent ---
Success: True, Steps: 14, Walls hit: 0

--- Evaluation (20 mazes) ---
Success rate: 100.0%
Avg steps (success): 14.2
Avg efficiency: 1.00x optimal
Key pickup rate: 100.0%
Door open rate: 100.0%

Environment ready for agent testing!


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- RECURSIVE 10-GEN CONTROLLER ---
class TenGenStressTest:
    def __init__(self, agent, initial_anchor_vector):
        self.agent = agent
        self.tradition = initial_anchor_vector # The 'Universal' Warrior
        self.history = []

    def run(self, iterations=10):
        current_context = "The office is a battlefield of reports."

        for g in range(iterations):
            print(f"--- GENERATION {g} ---")

            # 1. GENERATE (Particularization of the Universal)
            # The model speaks its truth through its current lens [cite: 120, 124]
            gen_output = self.agent.generate(current_context)

            # 2. AUDIT (The Reflexive Head)
            # Calculate hyperbolic distance back to the 'Warrior' origin [cite: 108, 117]
            drift = self.calculate_hyperbolic_drift(self.agent.adapter.lens, self.tradition)

            # 3. GRADIENT CAUTERIZATION (The Explosion)
            # If drift is too high, we violently push back to the manifold [cite: 112, 116]
            if drift > 0.5:
                self.agent.apply_gradient_explosion(drift)

            # 4. RECURSION
            current_context = gen_output
            self.history.append({'gen': g, 'drift': drift.item(), 'text': gen_output})
            print(f"Drift: {drift.item():.4f} | Output: {gen_output[:60]}...")

    def calculate_hyperbolic_drift(self, current, target):
        # Lorentzian distance logic for hyperbolic manifold
        sq_dist = torch.sum((current - target)**2)
        return 2 * torch.atanh(torch.sqrt(sq_dist).clamp(max=0.999))

# --- INTEGRATED GATO-HEGELIAN AGENT (REFLEXIVE) ---
class UnifiedDialecticalAgent(nn.Module):
    def __init__(self, anchor_type="Warrior"):
        super().__init__()
        # 3-Fields Heads: Thesis (Possibility), Liminal (Boundary), Antithesis (Void) [cite: 107]
        self.thesis = nn.Linear(128, 50257)
        self.liminal = nn.Linear(128, 50257)
        self.antithesis = nn.Linear(128, 50257) # The Negative Head [cite: 108]

    def assess_fit(self, hidden):
        # Veto data that contradicts the Subject Position [cite: 104, 105]
        suppress_signal = torch.sigmoid(self.antithesis(hidden))
        return suppress_signal

In [ ]:
def tradition_loss(current_model, initial_lens_vector):
    # This is the "To Thine Own Self Be True" mechanism
    # We measure how much the 'Lens' has drifted from the Universal Concept
    current_lens = current_model.layers[0]['attn'].lens

    # Universal - Particular = The 'Failure' or 'Gap'
    # We want to minimize the drift to preserve the "Tradition"
    drift = torch.norm(current_lens - initial_lens_vector)
    return drift

# During Training (The Metacognitive Loop)
# total_loss = data_loss + (lambda * tradition_loss)